# Optical vibrometry — demo

End-to-end demonstration of the `vibrometry` package (see `README.md` for the
module → TCC-section map).

**Part 1** verifies the pipeline on a synthetic free-decay video with known
ground truth (Phase-1 validation strategy applied to the software itself).

**Part 2** is the template for the real ruler experiment — fill in the video
path, calibration and measured ruler geometry.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # make Code/ importable

import matplotlib.pyplot as plt
import numpy as np

from vibrometry import (
    ROI, CantileverBeam, PipelineConfig, VibrometryPipeline,
    load_rois, load_video, save_rois, scale_from_pixels,
    select_rois_interactive,
)
from vibrometry import plotting
from vibrometry.synthetic import generate_cantilever_video

## Part 1 — Self-verification on a synthetic ground-truth video

A cantilever scene is rendered with an imposed free decay
$y(t) = A\,e^{-\zeta\omega_n t}\sin(\omega_d t)$ ($f_n$ = 2.5 Hz,
$\zeta$ = 0.012, $A$ = 60 px), so every pipeline output can be compared
against exact values.

In [ ]:
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

truth = generate_cantilever_video(out_dir / "synthetic_cantilever.mp4")
video = load_video(out_dir / "synthetic_cantilever.mp4")

tx, ty = truth.marker_center
r = truth.marker_radius + 14
sx, sy = truth.static_marker_center
rois = [
    ROI("tip", tx - r, ty - r, 2 * r, 2 * r),
    ROI("reference", sx - 26, sy - 26, 52, 52, is_reference=True),
]

pipeline = VibrometryPipeline(video, rois, PipelineConfig())
pipeline.run()
print(pipeline.report())
print(f"\nGround truth: f_d = {truth.f_d:.3f} Hz, zeta = {truth.zeta:.4f}")

In [ ]:
plotting.plot_tracking_overview(video.reference_frame, pipeline.tracks)
plotting.plot_time_series(video.times, pipeline.signals)
freqs, amp = pipeline.spectra["tip"]
plotting.plot_spectrum(freqs, amp, pipeline.modal_results[0].peaks)
if pipeline.decays["tip"] is not None:
    plotting.plot_decay_fit(video.times, pipeline.signals["tip"],
                            pipeline.decays["tip"])
plt.show()

In [ ]:
# sub-pixel tracking accuracy against the exact imposed trajectory
measured = pipeline.signals["tip"]
exact = truth.tip_y - truth.tip_y.mean()
err = measured - exact
print(f"RMS tracking error: {np.sqrt((err ** 2).mean()):.3f} px "
      f"(amplitude {np.abs(exact).max():.1f} px)")

## Part 2 — Real experiment (ruler as cantilever)

Steps:
1. copy the recorded video into `Code/videos/` and set `VIDEO_PATH`;
2. set `FPS_OVERRIDE` if the capture rate differs from the file metadata
   (e.g. 240 fps footage tagged for 30 fps playback);
3. measure a known length in the image to get the scale factor
   $s = d_{ref}/d_{px}$ [mm/pixel];
4. measure the ruler free length $L$, width $b$ and thickness $h$ and pick
   the material, for the analytical validation ($\varepsilon < 5\%$).

The first run opens an interactive window to draw the ROIs (tip marker
first, then a static background region) and saves them to `rois.json` for
later runs.

In [ ]:
VIDEO_PATH = "../videos/ruler.mp4"   # <-- the real recording
FPS_OVERRIDE = None                   # e.g. 240.0 for slow-motion footage
SCALE_MM_PER_PX = None                # e.g. scale_from_pixels(100.0, 222.0)
ROIS_JSON = Path("rois.json")

# measured ruler geometry [m] and material ('steel' or 'polypropylene')
beam = CantileverBeam.from_material(
    length=0.25, width=0.026, thickness=0.001, material="steel",
)
print(f"Analytical f_n1 = {beam.natural_frequency(1):.2f} Hz")

if Path(VIDEO_PATH).exists():
    video = load_video(VIDEO_PATH, fps_override=FPS_OVERRIDE)
    if ROIS_JSON.exists():
        rois = load_rois(ROIS_JSON)
    else:
        rois = select_rois_interactive(video.reference_frame)
        save_rois(rois, ROIS_JSON)

    pipeline = VibrometryPipeline(video, rois, PipelineConfig(
        scale_mm_per_px=SCALE_MM_PER_PX, beam=beam,
    ))
    pipeline.run()
    print(pipeline.report())
    pipeline.export("outputs/real")
else:
    print(f"Video not found: {VIDEO_PATH} — record/copy it first.")